In [5]:
import os

import torch
import pandas as pd
from dotenv import load_dotenv
from pyannote.audio import Pipeline

/home/movemin/venvs/AI_Agent/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
# 프로젝트 경로 설정
AUDIO_PATH = "./audio/싼기타_비싼기타.wav"

# 환경변수 불러오기
load_dotenv()
token = os.getenv("gpt-text-to-image")

In [7]:
if not token:
    raise RuntimeError("HUGGING_FACE_TOKEN이 없습니다.")


In [8]:
# 1. 모델 불러오기
print("화자 분리 모델 로딩 중...")

화자 분리 모델 로딩 중...


In [9]:
# instantiate the pipeline
pipeline = Pipeline.from_pretrained(
  "pyannote/speaker-diarization-community-1",
  token=token)

# run the pipeline on an audio file
output = pipeline(AUDIO_PATH)

# 실제 화자 분리 결과 가져오기
diarization = output.speaker_diarization


/home/movemin/venvs/AI_Agent/lib/python3.12/site-packages/pyannote/audio/models/blocks/pooling.py:103: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /__w/pytorch/pytorch/aten/src/ATen/native/ReduceOps.cpp:1927.)
  std = sequences.std(dim=-1, correction=1)


In [10]:

# dump the diarization output to disk using RTTM format
with open("./audio/싼기타_비싼기타.rttm", "w", encoding='utf-8') as rttm:
    diarization.write_rttm(rttm)

print("화자 분리 완료!")

화자 분리 완료!


In [ ]:
# RTTM을 CSV로 변환하기
import pandas as pd
rttm_path = "./audio/싼기타_비싼기타.rttm"

df_rttm = pd.read_csv(
    rttm_path,   # rttm 파일 경로
    sep=' ',     # 구분자는 띄어쓰기
    header=None, # 헤더는 없음
    names=['type', 'file', 'chnl', 'start', 'duration', 'C1', 'C2', 'speaker_id', 'C3', 'C4']
)

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN
1,SPEAKER,싼기타_비싼기타,1,7.405,3.949,NaN,NaN,SPEAKER_01,NaN,NaN
2,SPEAKER,싼기타_비싼기타,1,11.759,4.911,NaN,NaN,SPEAKER_01,NaN,NaN
3,SPEAKER,싼기타_비싼기타,1,17.227,10.648,NaN,NaN,SPEAKER_01,NaN,NaN
4,SPEAKER,싼기타_비싼기타,1,28.668,1.519,NaN,NaN,SPEAKER_01,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...
82,SPEAKER,싼기타_비싼기타,1,414.481,2.970,NaN,NaN,SPEAKER_00,NaN,NaN
83,SPEAKER,싼기타_비싼기타,1,417.755,3.476,NaN,NaN,SPEAKER_01,NaN,NaN
84,SPEAKER,싼기타_비싼기타,1,423.644,0.793,NaN,NaN,SPEAKER_00,NaN,NaN
85,SPEAKER,싼기타_비싼기타,1,424.741,3.544,NaN,NaN,SPEAKER_00,NaN,NaN


In [ ]:
# 발언이 끝난 시간 추가하기
df_rttm['end'] =  df_rttm['start'] + df_rttm['duration']

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798
1,SPEAKER,싼기타_비싼기타,1,7.405,3.949,NaN,NaN,SPEAKER_01,NaN,NaN,11.354
2,SPEAKER,싼기타_비싼기타,1,11.759,4.911,NaN,NaN,SPEAKER_01,NaN,NaN,16.670
3,SPEAKER,싼기타_비싼기타,1,17.227,10.648,NaN,NaN,SPEAKER_01,NaN,NaN,27.875
4,SPEAKER,싼기타_비싼기타,1,28.668,1.519,NaN,NaN,SPEAKER_01,NaN,NaN,30.187
...,...,...,...,...,...,...,...,...,...,...,...
82,SPEAKER,싼기타_비싼기타,1,414.481,2.970,NaN,NaN,SPEAKER_00,NaN,NaN,417.451
83,SPEAKER,싼기타_비싼기타,1,417.755,3.476,NaN,NaN,SPEAKER_01,NaN,NaN,421.231
84,SPEAKER,싼기타_비싼기타,1,423.644,0.793,NaN,NaN,SPEAKER_00,NaN,NaN,424.437
85,SPEAKER,싼기타_비싼기타,1,424.741,3.544,NaN,NaN,SPEAKER_00,NaN,NaN,428.285


In [ ]:
# 연속된 발화를 기록하기 위해 number 변수 추가하기
df_rttm['number'] = None
df_rttm.at[0, 'number'] = 0

display(df_rttm)

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.949,NaN,NaN,SPEAKER_01,NaN,NaN,11.354,None
2,SPEAKER,싼기타_비싼기타,1,11.759,4.911,NaN,NaN,SPEAKER_01,NaN,NaN,16.670,None
3,SPEAKER,싼기타_비싼기타,1,17.227,10.648,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,None
4,SPEAKER,싼기타_비싼기타,1,28.668,1.519,NaN,NaN,SPEAKER_01,NaN,NaN,30.187,None
...,...,...,...,...,...,...,...,...,...,...,...,...
82,SPEAKER,싼기타_비싼기타,1,414.481,2.970,NaN,NaN,SPEAKER_00,NaN,NaN,417.451,None
83,SPEAKER,싼기타_비싼기타,1,417.755,3.476,NaN,NaN,SPEAKER_01,NaN,NaN,421.231,None
84,SPEAKER,싼기타_비싼기타,1,423.644,0.793,NaN,NaN,SPEAKER_00,NaN,NaN,424.437,None
85,SPEAKER,싼기타_비싼기타,1,424.741,3.544,NaN,NaN,SPEAKER_00,NaN,NaN,428.285,None


In [ ]:
# 화자 번호 매기기
for i in range(1, len(df_rttm)):
    if df_rttm.at[i, "speaker_id"] != df_rttm.at[i - 1, "speaker_id"]:
        df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"] + 1
    else:
        df_rttm.at[i, "number"] = df_rttm.at[i - 1, "number"]

display(df_rttm.head(10))

,type,file,chnl,start,duration,C1,C2,speaker_id,C3,C4,end,number
0,SPEAKER,싼기타_비싼기타,1,0.993,5.805,NaN,NaN,SPEAKER_01,NaN,NaN,6.798,0
1,SPEAKER,싼기타_비싼기타,1,7.405,3.949,NaN,NaN,SPEAKER_01,NaN,NaN,11.354,0
2,SPEAKER,싼기타_비싼기타,1,11.759,4.911,NaN,NaN,SPEAKER_01,NaN,NaN,16.670,0
3,SPEAKER,싼기타_비싼기타,1,17.227,10.648,NaN,NaN,SPEAKER_01,NaN,NaN,27.875,0
4,SPEAKER,싼기타_비싼기타,1,28.668,1.519,NaN,NaN,SPEAKER_01,NaN,NaN,30.187,0
5,SPEAKER,싼기타_비싼기타,1,32.414,0.742,NaN,NaN,SPEAKER_00,NaN,NaN,33.156,1
6,SPEAKER,싼기타_비싼기타,1,33.562,3.527,NaN,NaN,SPEAKER_00,NaN,NaN,37.089,1
7,SPEAKER,싼기타_비싼기타,1,37.645,3.729,NaN,NaN,SPEAKER_00,NaN,NaN,41.374,1
8,SPEAKER,싼기타_비싼기타,1,41.628,1.080,NaN,NaN,SPEAKER_00,NaN,NaN,42.708,1
9,SPEAKER,싼기타_비싼기타,1,41.662,0.776,NaN,NaN,SPEAKER_01,NaN,NaN,42.438,2


In [21]:
# 같은 화자끼리 묶어서 정리하기
df_rttm_grouped = df_rttm.groupby("number").agg(
    start = pd.NamedAgg(column = "start", aggfunc = "min"),
    end = pd.NamedAgg(column = "end", aggfunc = "max"),
    speaker_id = pd.NamedAgg(column = "speaker_id", aggfunc = "first")
)

display(df_rttm_grouped)

,start,end,speaker_id
number,,,
0,0.993,30.187,SPEAKER_01
1,32.414,42.708,SPEAKER_00
2,41.662,44.024,SPEAKER_01
3,45.813,67.092,SPEAKER_00
4,67.227,82.769,SPEAKER_01
5,84.659,102.547,SPEAKER_00
6,103.492,117.515,SPEAKER_01
7,119.742,138.676,SPEAKER_00
8,139.351,168.967,SPEAKER_01


In [22]:

# 발화 시간 추가하고 인덱스 제거하기
df_rttm_grouped["duration"] = df_rttm_grouped["end"] - df_rttm_grouped["start"]
df_rttm_grouped = df_rttm_grouped.reset_index(drop = True)
display(df_rttm_grouped)


,start,end,speaker_id,duration
0,0.993,30.187,SPEAKER_01,29.194
1,32.414,42.708,SPEAKER_00,10.294
2,41.662,44.024,SPEAKER_01,2.362
3,45.813,67.092,SPEAKER_00,21.279
4,67.227,82.769,SPEAKER_01,15.542
5,84.659,102.547,SPEAKER_00,17.888
6,103.492,117.515,SPEAKER_01,14.023
7,119.742,138.676,SPEAKER_00,18.934
8,139.351,168.967,SPEAKER_01,29.616
9,170.924,192.304,SPEAKER_00,21.380


In [ ]:
# 화자 분리 결과를 CSV 파일로 저장하기
df_rttm_grouped.to_csv(
    "./audio/싼기타_비싼기타_rttm.csv",
    sep = ' ',
    index = False
)